# finance / economy / rates

**Intent**
- Outputs `finance_economy_rates_inflation_{low,base,high}_pct`, the inflation rate.
- Outputs `finance_economy_rates_riskfree_{low,base,high}_pct`, the nominal bank rate.
- Outputs `finance_economy_rates_real_{low,base,high}_pct`, the real return of cash.
- Gives low, base and high scenarios to 2070, in percent per year, as constants.
- Real equals risk-free less inflation, from the same `interest` rows.
- Is the floor and discount rate of `portfolio/return` and `portfolio/withdrawal`, and the inflation under `household/costs`.

**Data**
- `training` is the 5, 10, 20 and 40 year window means of the bank rate, inflation and real return of cash.
- It is synthetic until the `interest` table is read.
- That table holds the Reserve Bank `Bank`, `Inflation` and `Net` means over 1, 5, 10, 20 and 40 years.
- `golden` is one valid request, then a request with the input missing.

**Status**
- Low, base and high are placeholders, the smallest, the mean and the largest of the window means.
- The Fit cell asserts that real equals risk-free less inflation.
- It carries no evaluation, so the golden replay is the gate.

**Future**
- Overall: the work still to do before the first release:
  - Add `query_postgres` and read the `Bank` and `Net` windows.
  - Model inflation as an AR(1) pulled toward an anchored Reserve Bank target, with a de-anchored regime for the high scenario.
  - Model the real rate as mean reverting and add inflation, so the pair stays consistent.
  - Simulate to 2070, take compound rates, and set low, base and high as percentiles.
  - Compare the cash rate earned with the published one, and add index linked bond yields as an anchor.
  - Serve the numbers to the planning sheet through `wrangle` and an Apps Script function, which is not yet built.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "finance/economy/rates"
MODEL_NAME = "finance_economy_rates"
TARGET = {"kind": "served"}
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "economy_rates_training_2026-11-05_2026-11-05"
GOLDEN = "economy_rates_golden"

## Capture Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-capture-data))

In [ ]:
WINDOWS = [5, 10, 20, 40]
BANK = [2.0, 2.6, 3.6, 5.0]
INFLATION = [2.9, 2.7, 2.8, 3.24]

training_pl, training_path = ensure_data(
    WORKSPACE,
    "training",
    TRAINING,
    lambda: {"window_years": WINDOWS, "bank_pct": BANK, "inflation_pct": INFLATION, "net_pct": [round(bank - inflation, 4) for bank, inflation in zip(BANK, INFLATION, strict=True)]},
)
print("windows", training_pl.height)

## Fit ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-fit))

In [ ]:
bank, inflation, net = (training_pl[column].to_numpy() for column in ("bank_pct", "inflation_pct", "net_pct"))
assert abs(bank - inflation - net).max() < 1e-9, "the real return of cash is not the bank rate less inflation"


def scenarios(series):
    return [round(float(series.min()), 2), round(float(series.mean()), 2), round(float(series.max()), 2)]


CONSTANTS = dict(zip(OUTPUTS, [*scenarios(inflation), *scenarios(bank), *scenarios(net)], strict=True))
print("constants", CONSTANTS)
assert list(CONSTANTS.values()) == [2.7, 2.91, 3.24, 2.0, 3.3, 5.0, -0.9, 0.39, 1.76], "the fit did not recover the window summary"

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUTS[0]: [1.0, None], "expect": ["ok", "error"]})

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic window means of the bank rate, inflation and real cash return, training split",
        [training_path, golden_path],
        "smallest, mean and largest of the window means",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGET)
print("replayed the golden sequences exactly through the shipped loader")

## Register ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-register))

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGET, metadata["version"])
print("registered version", version)